## Bronze
### Ingestão e Auditoria Inicial dos Dados

*Case Técnico — Engenharia de Dados · iFood*

---

**Objetivo**

Implementar a camada Bronze do Lakehouse, responsável pelo armazenamento dos dados brutos
disponibilizados, preservando integralmente os arquivos de origem.

**Fonte dos dados**

| | |
|---|---|
| Dataset | NYC TLC Trip Record Data |
| Período | Janeiro a Maio de 2023 |
| Serviços | Yellow Cab, Green Cab |
| Formato | Parquet |

**Validações**

- Leitura individual dos arquivos
- Validação da quantidade de registros
- Validação do schema
- Identificação de divergências de tipos entre arquivos



**1. Instalando e carregando os pacotes**  

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row


**2. Leitura dos arquivos**  
- Leitura dos 10 arquivos;
- Confirmação dos nomes.


In [0]:
BRONZE_PATH = "/Volumes/ifood_case/bronze/raw"

files = dbutils.fs.ls(BRONZE_PATH)
display(files)

**3. Validação da carga**

Mostrar:
- Quantidade de registros por arquivo;
- Quantidade de colunas;
- Identificação do arquivo.
- Separa arquivos lidos em categorias: Green e Yellow.

In [0]:
BRONZE_PATH = "/Volumes/ifood_case/bronze/raw"

files = sorted([
    f.path for f in dbutils.fs.ls(BRONZE_PATH)
    if f.path.endswith(".parquet")
])

summary_df = None

for path in files:
    df = spark.read.parquet(path)
    row = df.select(
        F.lit(path).alias("file"),
        F.count(F.lit(1)).alias("rows"),
        F.lit(len(df.columns)).alias("columns")
    )
    summary_df = row if summary_df is None else summary_df.unionByName(row)

display(summary_df)

In [0]:
spark.read.parquet(files[0]).printSchema()

In [0]:
# Filtrando dinamicamente a partir da lista já coletada
green_files = [f for f in files if "green_tripdata" in f]
yellow_files = [f for f in files if "yellow_tripdata" in f]

**4. Auditoria dos Schemas**

Durante a validação foi identificado que arquivos de um mesmo serviço apresentam divergências de schema entre os meses, impedindo sua leitura consolidada pelo Spark.

Identificar:
- Colunas e períodos onde há divergência entre datatype;
- Toda padronização de schema será realizada na camada Silver (casts de tipos, padronização de nomenclatura e consolidação entre Yellow e Green Cab).

In [0]:
green_columns = [
    "VendorID",
    "passenger_count",
    "total_amount",
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "RatecodeID",
    "store_and_fwd_flag", 
    "payment_type"
]

yellow_columns = [
    "VendorID",
    "passenger_count",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type"
]

In [0]:
def auditar_schema(files, taxi_type, columns):
    linhas = []

    for file in files:
        periodo = file.split("_2023-")[1].replace(".parquet", "")
        df = spark.read.parquet(file)

        total_registros = df.count()

        # Monta todas as expressões de agregação de uma vez,
        # para todas as colunas, executando em uma única ação Spark
        agg_exprs = []
        for coluna in columns:
            agg_exprs += [
                F.min(coluna).alias(f"{coluna}__min"),
                F.max(coluna).alias(f"{coluna}__max"),
                F.count(F.when(F.col(coluna).isNull(), 1)).alias(f"{coluna}__nulls"),
            ]

        stats_row = df.select(*agg_exprs).collect()[0]

        for coluna in columns:
            campo = df.schema[coluna]

            linhas.append(Row(
                taxi_type=taxi_type,
                periodo=f"2023-{periodo}",
                total_registros=total_registros,
                coluna=coluna,
                datatype=str(campo.dataType),
                nullable=campo.nullable,
                min=str(stats_row[f"{coluna}__min"]),
                max=str(stats_row[f"{coluna}__max"]),
                nulls=stats_row[f"{coluna}__nulls"]
            ))

    df_audit = spark.createDataFrame(linhas)

    colunas_divergentes = (
        df_audit
        .groupBy("taxi_type", "coluna")
        .agg(F.countDistinct("datatype").alias("qtd_datatypes"))
        .filter(F.col("qtd_datatypes") > 1)
        .select("taxi_type", "coluna")
    )

    return (
        df_audit
        .join(colunas_divergentes, ["taxi_type", "coluna"], "inner")
        .orderBy("taxi_type", "coluna", "periodo")
    )

In [0]:
audit_green = auditar_schema(
    green_files,
    "green",
    green_columns
)

audit_yellow = auditar_schema(
    yellow_files,
    "yellow",
    yellow_columns
)

In [0]:
display(audit_green)

In [0]:
display(audit_yellow)